# Module 10.2: Speculative Decoding

As we established in previous modules, generating text with LLMs requires "autoregressive generation." You feed the prompt in, get the next token, append it, feed it ALL back in, get the next one, append it, feed it ALL back in. 

This means if we generate 100 new words, we must load the massive model weights into the GPU compute cores 100 separate times! This makes LLM inference **Memory Bandwidth Bound**, not Compute Bound. Our extremely fast GPU cores are spending 99% of their time waiting for the weights to be loaded from VRAM.

## 1. The Core Idea: The Manager and the Assistant

### The Analogy
Imagine a brilliant Software Architect (the 70B parameter Target Model) writing code. They spend 10 minutes thinking, then write 1 single line of code. They are perfect, but slow.

Now, imagine the Architect hires a Junior Developer (a 1B parameter Draft Model). The Junior Dev types incredibly fast but makes mistakes. 
The Junior Dev aggressively types out 5 lines of code in 10 seconds. 
The Architect then reads all 5 lines *at the same time* (in parallel). 
The Architect says: *"Line 1 is good, Line 2 is good, Line 3 is good, but Line 4 is a bug. Keep the first 3 lines, throw away the rest, and I will write Line 4 myself."*

**This is Speculative Decoding!**
- The **Draft Model** quickly predicts 5 future tokens.
- The **Target Model** evaluates all 5 tokens simultaneously in one single forward pass.
- If the fast model guessed right, we get multiple tokens for the "cost" of one model load. If not, we fall back instantly to the Target Model's output.

## 2. Implementing the Concept

Because evaluating a sequence of known tokens in parallel is mathematically identical to a normal prompt processing pass, the Target model can verify all guesses at once. Let's mock this out.

In [1]:
import torch
import time

# Pretend we have a Vocabulary of 50,000 words
VOCAB_SIZE = 50000

def mock_draft_model(prompt_tokens, num_guesses=4):
    # The tiny draft model shoots out fast, slightly inaccurate guesses.
    # We'll just generate random tokens for this mock, assuming it takes 0.01 seconds.
    time.sleep(0.01) 
    return torch.randint(0, VOCAB_SIZE, (num_guesses,))

def mock_target_model(sequence_tokens):
    # The huge model evaluates the sequence. It takes 0.1 seconds to load its weights,
    # but thanks to parallel Attention, it evaluates N tokens basically instantly!
    time.sleep(0.1)
    # Standard next-token prediction logic: output logits for next tokens
    batch = 1
    seq_len = sequence_tokens.size(0)
    # Mock logits shape: [seq_len, VOCAB_SIZE]. Each position predicts the token *after* it.
    mock_logits = torch.randn(seq_len, VOCAB_SIZE) 
    
    # We do a greedy argmax to find the model's true target predictions
    target_predictions = torch.argmax(mock_logits, dim=-1)
    return target_predictions


## 3. The Verification Logic

We compare the Junior's guesses against the Architect's true predictions.

In [2]:
def speculative_decode_step(prompt_seq, num_drafts=4):
    print(f"\n--- Starting Speculative Step ---")
    print(f"Original Prompt Length Context: {len(prompt_seq)}")
    
    # 1. Draft model guesses 4 tokens ahead
    draft_guesses = mock_draft_model(prompt_seq, num_drafts)
    print(f"Draft Guesses: {draft_guesses.tolist()}")
    
    # 2. Append guesses to prompt temporarily
    speculative_seq = torch.cat([prompt_seq, draft_guesses])
    
    # 3. Target Model parallel evaluation!
    # This one pass checks the last token in prompt + the 4 draft guesses
    target_preds = mock_target_model(speculative_seq)
    
    # Extract predictions corresponding to the drafted positions
    # target_preds[-5:-1] checks the draft guesses. target_preds[-1] is what to do next.
    eval_preds = target_preds[-(num_drafts+1):-1]
    target_next = target_preds[-1:]
    
    # 4. Acceptance Check
    accepted_count = 0
    for i in range(num_drafts):
        if draft_guesses[i] == eval_preds[i]:
            accepted_count += 1
        else:
            # Divergence! Stop accepting immediately.
            break
            
    if accepted_count > 0:
        print(f"SUCCESS: Target agreed with {accepted_count} guesses!")
        accepted_tokens = draft_guesses[:accepted_count]
    else:
        print("REJECTED: Target disagreed with the first guess.")
        accepted_tokens = torch.tensor([], dtype=torch.long)
    
    # 5. Append the accepted drafts + the target's correction/continuation
    # We always get at least ONE token from the Target model (eval_preds[accepted_count])
    correction_token = eval_preds[accepted_count].unsqueeze(0) if accepted_count < num_drafts else target_next
    
    final_yield = torch.cat([accepted_tokens, correction_token])
    print(f"Yielding Tokens: {final_yield.tolist()} (Generated {len(final_yield)} tokens for 1 Target Pass!)")
    return final_yield


prompt = torch.tensor([5, 12, 59, 1002]) # Mock prompt

# Simulating a loop. In reality, the draft model gets accepted a decent % of the time.
for step in range(3):
    new_tokens = speculative_decode_step(prompt, num_drafts=4)
    prompt = torch.cat([prompt, new_tokens])

## 4. Why does this heavily accelerate LLMs?
If the Draft Model is moderately good (even just 40-50% accuracy on predicting the exact token), we average generating ~2.5 tokens per Target Model evaluation.

Because the Target Model is bottlenecked by RAM speed (loading weights), doing a forward pass for 1 token takes nearly the exact same amount of time as a forward pass for 5 tokens. Thus, Speculative Decoding gives us a literal free 2x-3x speedup on inference, maintaining **perfect identical output quality** to running only the large model!